# EchoFactory NB-03: Training STgram-MFN
Arsitektur: MobileFaceNet backbone (dual-branch) + ArcFace Loss

**GPU**: Aktifkan T4 di Settings -> Accelerator -> GPU T4 x1

## Cara pakai:
1. Set `MACHINE_TYPE` di Cell 2 (`fan` / `pump` / `slider` / `valve`)
2. Run All
3. Model tersimpan ke `/kaggle/working/stgram_mfn_{machine}.pt`
4. Ulangi dengan machine type berbeda


In [ ]:
import os
import gc
import json
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'VRAM: {vram:.1f} GB')
    print('Mixed Precision FP16: AKTIF')


In [ ]:
# ========== UBAH DI SINI ==========
MACHINE_TYPE = 'fan'   # 'fan' | 'pump' | 'slider' | 'valve'
# ====================================

FEAT_DIR = '/kaggle/working/features'
BATCH_SIZE = 64
EPOCHS = 100
LR = 1e-3
WEIGHT_DECAY = 5e-4
EMBED_DIM = 256
ARC_S = 12.0
ARC_M = 0.3
WARMUP_EPOCHS = 10
OUT_MODEL = f'/kaggle/working/stgram_mfn_{MACHINE_TYPE}.pt'

print(f'Machine  : {MACHINE_TYPE.upper()}')
print(f'Output   : {OUT_MODEL}')
print(f'Epochs   : {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR}')
print(f'ArcFace  : s={ARC_S}, m={ARC_M} | Warmup: {WARMUP_EPOCHS} epochs')


In [ ]:
class MIMIIDataset(Dataset):
    def __init__(self, machine, feat_dir, cond='normal'):
        path = os.path.join(feat_dir, f'{machine}_{cond}.pt')
        data = torch.load(path)
        self.feats = data['features']
        self.labels = data['labels']
        self.n_cls = int(self.labels.max().item()) + 1
        print(f'Loaded {machine}/{cond}: N={len(self.feats)}, n_classes={self.n_cls}')

    def __len__(self):
        return len(self.feats)

    def __getitem__(self, idx):
        f = self.feats[idx]
        return f[0:1], f[1:2], self.labels[idx]

train_ds = MIMIIDataset(MACHINE_TYPE, FEAT_DIR, 'normal')
N_CLASSES = train_ds.n_cls
train_dl = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True, persistent_workers=True
)
print(f'DataLoader: {len(train_dl)} batches | {N_CLASSES} machine IDs')


In [ ]:
class ConvBNPReLU(nn.Module):
    def __init__(self, ic, oc, k=3, s=1, p=1, g=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ic, oc, k, s, p, groups=g, bias=False),
            nn.BatchNorm2d(oc),
            nn.PReLU(oc)
        )

    def forward(self, x):
        return self.net(x)

class DepthwiseSep(nn.Module):
    def __init__(self, ic, oc, s=1):
        super().__init__()
        self.net = nn.Sequential(
            ConvBNPReLU(ic, ic, s=s, g=ic),
            ConvBNPReLU(ic, oc, k=1, p=0)
        )

    def forward(self, x):
        return self.net(x)

class MobileFaceNet(nn.Module):
    def __init__(self, ed=256):
        super().__init__()
        self.enc = nn.Sequential(
            ConvBNPReLU(1, 32, s=2),
            DepthwiseSep(32, 64),
            DepthwiseSep(64, 128, s=2),
            DepthwiseSep(128, 128),
            DepthwiseSep(128, 256, s=2),
            DepthwiseSep(256, 256),
            DepthwiseSep(256, 512, s=2),
            nn.AdaptiveAvgPool2d(1)
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, ed),
            nn.BatchNorm1d(ed)
        )

    def forward(self, x):
        return self.head(self.enc(x))

b = MobileFaceNet(EMBED_DIM)
o = b(torch.randn(2, 1, 128, 128))
print(f'MobileFaceNet output: {o.shape}')
print(f'Params: {sum(p.numel() for p in b.parameters()):,}')


In [ ]:
class ArcFace(nn.Module):
    def __init__(self, ed, nc, s=32.0, m=0.5):
        super().__init__()
        self.s = s
        self.m = m
        self.W = nn.Parameter(torch.FloatTensor(nc, ed))
        nn.init.xavier_uniform_(self.W)
        self.cos_m = math.cos(m)
        self.sin_m = math.sin(m)
        self.th = math.cos(math.pi - m)
        self.mm = math.sin(math.pi - m) * m

    def forward(self, feat, labels):
        cos = F.normalize(feat, 1) @ F.normalize(self.W, 1).T
        sin = (1.0 - cos ** 2 + 1e-8).sqrt()
        phi = cos * self.cos_m - sin * self.sin_m
        phi = torch.where(cos > self.th, phi, cos - self.mm)
        one_hot = F.one_hot(labels, cos.shape[1]).float()
        out = (one_hot * phi + (1.0 - one_hot) * cos) * self.s
        return F.cross_entropy(out, labels)

print('ArcFace defined OK')


In [ ]:
class STgramMFN(nn.Module):
    def __init__(self, nc, ed=256):
        super().__init__()
        self.mel = MobileFaceNet(ed)
        self.tgram = MobileFaceNet(ed)
        self.fuse = nn.Sequential(
            nn.Linear(ed * 2, ed),
            nn.BatchNorm1d(ed),
            nn.PReLU(ed)
        )
        self.arc = ArcFace(ed, nc, ARC_S, ARC_M)

    def forward(self, mel, tg, labels=None):
        feat = self.fuse(torch.cat([self.mel(mel), self.tgram(tg)], dim=1))
        feat = F.normalize(feat, dim=1)
        if labels is not None:
            return feat, self.arc(feat, labels)
        return feat

model = STgramMFN(N_CLASSES, EMBED_DIM).to(device)
tp = sum(p.numel() for p in model.parameters())
print(f'STgram-MFN on {device} | {tp:,} params ({tp/1e6:.2f}M)')
with torch.no_grad():
    dm = torch.randn(4, 1, 128, 128).to(device)
    dt = torch.randn(4, 1, 128, 128).to(device)
    dl = torch.randint(0, N_CLASSES, (4,)).to(device)
    f, l = model(dm, dt, dl)
    print(f'Forward test OK: feat={f.shape}, loss={l.item():.4f}')


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Scheduler: 10 epochs linear warmup + Cosine Annealing
scheduler_warmup = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS, eta_min=1e-5)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[scheduler_warmup, scheduler_cosine], milestones=[WARMUP_EPOCHS])

scaler = GradScaler()

best_loss = float('inf')
losses = []
t0 = time.time()

print(f'Training {MACHINE_TYPE.upper()} | {EPOCHS} epochs | batch={BATCH_SIZE}')
print('-' * 55)

for ep in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0

    for mel, tg, lab in train_dl:
        mel = mel.to(device, non_blocking=True)
        tg = tg.to(device, non_blocking=True)
        lab = lab.to(device, non_blocking=True)

        optimizer.zero_grad()
        with autocast():
            _, loss = model(mel, tg, lab)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()

    avg = epoch_loss / len(train_dl)
    losses.append(avg)
    scheduler.step()

    if avg < best_loss:
        best_loss = avg
        torch.save({
            'epoch': ep,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'best_loss': best_loss,
            'machine': MACHINE_TYPE,
            'n_classes': N_CLASSES,
            'embed_dim': EMBED_DIM,
            'config': {'ARC_S': ARC_S, 'ARC_M': ARC_M}
        }, OUT_MODEL)
        tag = ' <- BEST'
    else:
        tag = ''

    if ep % 5 == 0 or ep == 1:
        elapsed = (time.time() - t0) / 60
        lr_now = scheduler.get_last_lr()[0]
        print(f'Ep {ep:3d}/{EPOCHS} | Loss: {avg:.4f} | LR: {lr_now:.1e} | {elapsed:.1f}min{tag}')

print(f'\nDone! Best loss: {best_loss:.4f} -> {OUT_MODEL}')
if torch.cuda.is_available():
    print(f'Peak VRAM: {torch.cuda.max_memory_allocated()/(1024**3):.2f} GB')


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(losses) + 1), losses, 'b-', lw=1.5, label='Train Loss')
plt.axhline(best_loss, color='r', ls='--', lw=1, label=f'Best={best_loss:.4f}')
plt.xlabel('Epoch')
plt.ylabel('ArcFace Loss')
plt.title(f'Training Curve - {MACHINE_TYPE.upper()} | STgram-MFN', fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'/kaggle/working/curve_{MACHINE_TYPE}.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'NB-03 {MACHINE_TYPE.upper()} SELESAI!')
print('-> Ulangi dengan MACHINE_TYPE lain, atau lanjut ke NB-04')
